# Document Extraction Deep Dive

This notebook explores different document extraction strategies using the ExtractOperator.

## Extraction Modes
1. **Basic Text Extraction** - Standard Docling library extraction
2. **VLM-Enhanced Extraction** - Vision-Language Model for complex layouts
3. **Entity Extraction** - Extract structured data with LLMs

## What You'll Learn
- How to choose the right extraction mode
- Configure extraction parameters
- Compare extraction quality
- Extract entities with custom schemas

## Prerequisites
- Sample PDFs in `tests/fixtures/invoices/`
- Ollama running (for entity extraction and VLM examples)
- Model `llama3.2` pulled: `ollama pull llama3.2`
- Model `ibm/granite-docling:258m` pulled (for VLM section): `ollama pull ibm/granite-docling:258m`

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import pandas as pd

# Add src to path if needed
import os
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("✓ Imports loaded successfully")

# Check if sample PDFs exist
invoice_dir = Path("../../tests/fixtures/invoices")
if invoice_dir.exists():
    pdfs = list(invoice_dir.glob("*.pdf"))
    print(f"✓ Found {len(pdfs)} sample PDF(s) in {invoice_dir}")
    for pdf in pdfs[:3]:  # Show first 3
        print(f"  - {pdf.name}")
else:
    print(f"✗ Sample PDF directory not found: {invoice_dir}")

### Helper Functions

These utility functions help display extraction results in a readable format:

- **`load_operator_results()`** - Loads operator output from filesystem storage (parquet files)
- **`display_extraction_result()`** - Shows extracted document content with metadata (name, pages, content preview)
- **`display_entities()`** - Formats extracted entities from JSON into readable output
- **`cleanup_flow_data()`** - Deletes local data directory after flow execution to save disk space

In [ ]:
# Helper Functions for Display
def load_operator_results(manager, operator_name="extract", operator_index=0):
    """
    Load operator output from filesystem storage.
    
    Args:
        manager: DocpipeFlowManager instance after execution
        operator_name: Name of the operator (default: 'extract')
        operator_index: Index of the operator if multiple exist (default: 0)
    
    Returns:
        PyArrow Table with operator output, or None if not found
    """
    import pyarrow.parquet as pq
    from pathlib import Path
    
    try:
        session_info = manager.session_info
        job_id = session_info.job_id
        job_run_id = session_info.job_run_id
        
        print(f"\nJob ID: {job_id}")
        print(f"Job Run ID: {job_run_id}")
        
        # Construct exact path to operator output
        # Structure: data/{job_id}/{job_run_id}/data/{operator_name}_{index}/output.parquet
        operator_dir = f"{operator_name}_{operator_index}"
        result_file = Path(f"./data/{job_id}/{job_run_id}/data/{operator_dir}/output.parquet")
        
        if result_file.exists():
            # Read the parquet file
            result_table = pq.read_table(result_file)
            print(f"\nLoaded results from: data/{operator_dir}/output.parquet")
            print(f"Rows: {len(result_table)}")
            print(f"Columns: {result_table.column_names}")
            return result_table
        print(f"\nFile not found: {result_file}")
        
        # Show available files for debugging
        data_dir = Path(f"./data/{job_id}/{job_run_id}")
        if data_dir.exists():
            print("\nAvailable files:")
            for f in data_dir.rglob("*.parquet"):
                print(f"  - {f.relative_to(data_dir)}")
        return None
            
    except Exception as e:
        print(f"\nError loading results: {e}")
        print("\nNote: Ensure 'data_storage_type': 'local' is set in global_config")
        return None


def display_extraction_result(result_table, max_content_length=500):
    """Display extraction results in a formatted way"""
    if result_table is None or len(result_table) == 0:
        print("No results to display")
        return
    
    from pathlib import Path
    
    # Summary statistics
    total_docs = len(result_table)
    total_pages = sum(result_table['pages_processed'].to_pylist()) if 'pages_processed' in result_table.column_names else 0
    
    print("\n" + "="*80)
    print("EXTRACTION RESULTS SUMMARY")
    print("="*80)
    print(f"Total Documents: {total_docs}")
    print(f"Total Pages: {total_pages}")
    print(f"Columns: {', '.join(result_table.column_names)}")
    
    # Show first 3 documents
    print("\n" + "-"*80)
    print("DOCUMENT PREVIEWS (showing first 3)")
    print("-"*80)
    
    for i in range(min(len(result_table), 3)):
        print(f"\n[Document {i+1}/{total_docs}]")
        
        # Show filename only (not full path)
        if 'name' in result_table.column_names:
            full_path = result_table['name'][i].as_py()
            filename = Path(full_path).name
            print(f"File: {filename}")
        
        # Show metadata
        if 'pages_processed' in result_table.column_names:
            pages = result_table['pages_processed'][i].as_py()
            print(f"Pages: {pages}")
        
        if 'document_format' in result_table.column_names:
            doc_format = result_table['document_format'][i].as_py()
            print(f"Format: {doc_format}")
        
        # Show content preview with better formatting
        if 'content' in result_table.column_names:
            content = result_table['content'][i].as_py()
            content_length = len(content)
            
            # Clean up content for display
            preview = content[:max_content_length]
            # Remove excessive newlines
            preview = '\n'.join(line for line in preview.split('\n') if line.strip())
            
            print(f"\nContent Preview ({len(preview)}/{content_length} chars):")
            print("-" * 40)
            print(preview)
            if content_length > max_content_length:
                print("...")
            print("-" * 40)
    
    print("\n" + "="*80)

def display_entities(entities_json_str):
    """Format entities as a nice table"""
    import json
    try:
        entities = json.loads(entities_json_str)
        if isinstance(entities, dict):
            print("\nExtracted Entities:")
            for key, value in entities.items():
                print(f"  {key}: {value}")
        elif isinstance(entities, list):
            print("\nExtracted Entities:")
            for i, entity in enumerate(entities, 1):
                print(f"  [{i}] {entity}")
    except json.JSONDecodeError:
        print(f"Raw entities: {entities_json_str}")


def cleanup_flow_data(manager):
    """Clean up local storage data after flow execution"""
    import shutil
    from pathlib import Path
    
    try:
        session_info = manager.session_info
        job_id = session_info.job_id
        
        # Path to the entire job_id folder (includes all job runs and metadata)
        data_dir = Path(f"./data/{job_id}")
        
        if data_dir.exists():
            shutil.rmtree(data_dir)
            print(f"\nCleaned up data directory: {data_dir}")
        else:
            print(f"\nNo data directory found to clean up")
    except Exception as e:
        print(f"\nError during cleanup: {e}")


print("Helper functions loaded successfully")

## 1. Basic Text Extraction

Standard extraction using Docling library - fast and reliable for most documents:

In [ ]:
basic_extraction_flow = {
    "flow_name": "basic-extraction",
    "description": "Basic text extraction with Docling",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True,
        "data_storage_type": "local"  # Store intermediate data on local filesystem
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/invoices"]},
                "include_filter": "pdf",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        }
    ]
}

print("Executing basic text extraction...")
print("Mode: Docling Library (standard)")
print()

manager = DocpipeFlowManager(flow_def=basic_extraction_flow)
manager.execute()

print("\n✓ Basic extraction completed!")
print("\nCharacteristics:")
print("  - Fast processing speed")
print("  - Good for standard documents")
print("  - Handles tables and structure")
print("  - No external API calls needed")

# Load and display extraction results using helper function
result_table = load_operator_results(manager, operator_name="extract", operator_index=0)
if result_table:
    display_extraction_result(result_table, max_content_length=300)

# Clean up stored data
cleanup_flow_data(manager)

## 2. VLM-Enhanced Extraction

Vision-Language Models (VLM) provide superior extraction quality for:
- Complex document layouts
- Tables with intricate structures
- Charts, diagrams, and infographics
- Scanned documents with poor OCR
- Mixed content (text + images)

**Note:** Requires Ollama with a vision model (e.g., `ibm/granite-docling:258m`)

In [ ]:
vlm_extraction_flow = {
    "flow_name": "vlm-enhanced-extraction",
    "description": "Extract with Vision-Language Model for complex layouts",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True,
        "data_storage_type": "local"
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/invoices"]},
                "include_filter": "pdf",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content",
                    "provider_config": {
                        "vlm_pipeline": {
                            "preset": "granite_docling",
                            "engine": "api_ollama",
                            "engine_options": {
                                "api_base": "http://localhost:11434/v1/chat/completions",
                                "model_id": "ibm/granite-docling:258m",
                                "request_timeout": 300
                            }
                        }
                    }
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        }
    ]
}

print("Executing VLM-enhanced extraction...")
print("Mode: Docling Library with IBM Granite Docling Vision Model")
print("\nVLM Benefits:")
print("  - Better handling of complex layouts")
print("  - Improved table structure recognition")
print("  - Enhanced chart/diagram extraction")
print("  - Better for scanned documents")
print("\nNote: Requires ibm/granite-docling:258m model")
print("      Install: ollama pull ibm/granite-docling:258m")
print()

try:
    manager = DocpipeFlowManager(flow_def=vlm_extraction_flow)
    manager.execute()
    
    print("\nVLM-enhanced extraction completed!")
    print("\nWhen to use VLM:")
    print("  - Documents with complex tables")
    print("  - Charts, diagrams, or infographics")
    print("  - Scanned documents with poor OCR")
    print("  - Handwritten notes or forms")
    print("  - Mixed content (text + images)")
    
    # Load and display extraction results using helper function
    result_table = load_operator_results(manager, operator_name="extract", operator_index=0)
    if result_table:
        display_extraction_result(result_table, max_content_length=1000)
    
    # Clean up stored data
    cleanup_flow_data(manager)
except Exception as e:
    print(f"\nVLM extraction failed: {e}")
    print("\nMake sure:")
    print("  1. Ollama is running: ollama serve")
    print("  2. Vision model is pulled: ollama pull ibm/granite-docling:258m")
    print("  3. Model is available: ollama list")

## 3. Entity Extraction (No Schema)

Extract entities from documents without defining a schema. The LLM will automatically identify and extract relevant entities.

**Key Points:**
- No predefined schema required
- LLM identifies entities automatically
- Good for exploratory analysis
- Flexible output structure

**Note:** Requires Ollama with an LLM (e.g., `llama3.2`)

In [ ]:
entity_extraction_no_schema_flow = {
    "flow_name": "entity-extraction-no-schema",
    "description": "Extract entities without predefined schema",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True,
        "data_storage_type": "local"
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/invoices"]},
                "include_filter": "pdf",

            }}
        },
        {
            "name": "classify",
            "type": "document_classifier",
            "depends_on": ["ingest"],
            "config": {
                "provider": "litellm",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/llama3.2",
                    "api_base": "http://localhost:11434",
                    "api_key": "<your-api-key>"
                }
            }
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["classify"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "litellm",
                    "provider_config": {
                        "model_id": "ollama/llama3.2",
                        "api_base": "http://localhost:11434",
                        "api_key": "<your-api-key>",
                        "temperature": 0.0,
                        "max_tokens": 2000
                    }
                }
            }
        }
    ]
}

print("Executing entity extraction without schema...")
print("Mode: LiteLLM with Ollama (No Schema)")
print("\nBenefits:")
print("  - No schema definition needed")
print("  - LLM identifies entities automatically")
print("  - Good for exploratory analysis")
print("  - Flexible output structure")
print("\nNote: This will take longer as it uses LLM for extraction")
print()

try:
    manager = DocpipeFlowManager(flow_def=entity_extraction_no_schema_flow)
    manager.execute()
    
    print("\n✓ Entity extraction (no schema) completed!")
    print("\nWhen to use:")
    print("  - Exploring new document types")
    print("  - Unknown entity structures")
    print("  - Rapid prototyping")
    print("  - Discovery phase of projects")
    
    # Load and display extraction results
    result_table = load_operator_results(manager, operator_name="extract", operator_index=0)
    if result_table:
        display_extraction_result(result_table, max_content_length=500)
        
        # Display entities if available
        if 'entities' in result_table.column_names and len(result_table) > 0:
            print("\n" + "="*80)
            print("EXTRACTED ENTITIES (First Document)")
            print("="*80)
            entities_json = result_table['entities'][0].as_py()
            if entities_json:
                display_entities(entities_json)
    
    # Clean up stored data
    cleanup_flow_data(manager)
except Exception as e:
    print(f"\n✗ Entity extraction failed: {e}")
    print("\nMake sure:")
    print("  1. Ollama is running: ollama serve")
    print("  2. Model is pulled: ollama pull llama3.2")

## 4. Structured Entity Extraction with Custom Schema

Extract structured data from documents using LLMs (Ollama) by defining a custom schema.

**Key Points:**
- Requires a `custom_schema` to define what entities to extract
- Uses LLM (via LiteLLM) to intelligently extract entities
- Returns structured JSON data matching your schema
- Ideal for production pipelines with known document types

In [ ]:
# Define a custom schema for invoice extraction
# Note: Schema is defined here for documentation, but must also be included
# inline in the flow config below as it's required by the operator

invoice_schema = {
    "invoice_number": "string",
    "invoice_date": "string",
    "total_amount": "string",
    "vendor_name": "string"
}

entity_extraction_flow = {
    "flow_name": "entity-extraction-with-schema",
    "description": "Extract entities with custom schema",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True,
        "data_storage_type": "local"
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/invoices"]},
                "include_filter": "pdf",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "litellm",
                    "custom_schema": {
                        "invoice_number": "string",
                        "invoice_date": "string",
                        "total_amount": "string",
                        "vendor_name": "string"
                        },
                    "expand_extracted_data": True,
                    "provider_config": {
                        "model_id": "ollama/llama3.2",
                        "api_base": "http://localhost:11434",
                        "api_key": "<your-api-key>"
                    }
                }
            }
        }
    ]
}

print("Executing entity extraction with custom schema...")
print("Mode: LiteLLM with Ollama")
print(f"\nSchema: {invoice_schema}")
print("Note: This will take longer as it uses LLM for extraction")
print()

try:
    manager = DocpipeFlowManager(flow_def=entity_extraction_flow)
    manager.execute()
    
    print("\n✓ Entity extraction completed!")
    print("\nBenefits of schema-based extraction:")
    print("  - Consistent output structure")
    print("  - Type validation")
    print("  - Better for downstream processing")
    print("  - Easier to integrate with databases")
    
    # Load and display extraction results
    result_table = load_operator_results(manager, operator_name="extract", operator_index=0)
    if result_table:
        display_extraction_result(result_table, max_content_length=300)
        
        # Display extracted entities if available
        if 'entities' in result_table.column_names:
            print("\n" + "="*80)
            print("EXTRACTED ENTITIES")
            print("="*80)
            for i in range(min(len(result_table), 3)):
                entities_json = result_table['entities'][i].as_py()
                if entities_json:
                    print(f"\n[Document {i+1}]")
                    display_entities(entities_json)
            print("\n" + "="*80)
    
    # Clean up stored data
    cleanup_flow_data(manager)
except Exception as e:
    print(f"\n✗ Entity extraction failed: {e}")
    print("\nMake sure:")
    print("  1. Ollama is running: ollama serve")
    print("  2. Model is pulled: ollama pull llama3.2")

## 5. Extraction Mode Comparison

### When to Use Each Mode

| Mode | Speed | Quality | Use Case |
|------|-------|---------|----------|
| **Basic (Docling Library)** | Fast | Good | Standard documents, bulk processing |
| **VLM-Enhanced** | Slow | Excellent | Complex layouts, tables, images |
| **Entity Extraction (No Schema)** | Medium | Very Good | Exploratory analysis |
| **Entity Extraction (With Schema)** | Medium | Excellent | Production pipelines |

### Recommendations

1. **Start with Basic** - Use Docling Library for initial testing
2. **Add VLM if needed** - For documents with complex layouts or poor OCR
3. **Use Entity Extraction** - When you need structured data
4. **Define Schemas** - For production use cases with consistent output requirements

## 6. Multi-Format Output



Extract documents in multiple formats simultaneously:

- **Markdown** (default): Best for text processing

- **HTML**: For web display

- **JSON**: For structured data

- **Text**: Plain text without formatting

- **DocTags**: Docling's native tag format with semantic structure

- **DocLang**: DocLang format for advanced document representation



Use the `additional_formats` parameter to specify extra output formats.


In [ ]:
multi_format_flow = {
    "flow_name": "multi-format-extraction",
    "description": "Extract in multiple formats",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True,
        "data_storage_type": "local"
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/invoices"]},
                "include_filter": "pdf",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content",
                    "provider_config": {
                        "additional_formats": ["html", "json", "text", "doctags", "doclang"]
                    }
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        }
    ]
}

print("Executing multi-format extraction...")
manager = DocpipeFlowManager(flow_def=multi_format_flow)
manager.execute()

# Load and display extraction results using helper function

result_table = load_operator_results(manager, operator_name="extract", operator_index=0)

if result_table:
    print("\n" + "="*80)
    print("OUTPUT FORMATS COMPARISON")
    print("="*80)
    
    # Show available columns
    format_columns = [col for col in result_table.column_names if col.startswith("content")]
    print(f"\nGenerated {len(format_columns)} format columns: {', '.join(format_columns)}")
    
    if len(result_table) > 0:
        print("\n" + "-"*80)
        print("FORMAT PREVIEWS (first 200 characters of each)")
        print("-"*80)
        
        # Markdown
        print("\n[1] MARKDOWN (default - best for text processing)")
        if 'content' in result_table.column_names:
            preview = result_table['content'][0].as_py()[:200]
            print(f"    {preview}...")
        print("\n" + "- " * 40)
        
        # HTML
        print("\n[2] HTML (for web display)")
        if 'content_html' in result_table.column_names:
            preview = result_table['content_html'][0].as_py()[:200]
            print(f"    {preview}...")
        print("\n" + "- " * 40)
        
        # Plain Text
        print("\n[3] PLAIN TEXT (no formatting)")
        if 'content_text' in result_table.column_names:
            preview = result_table['content_text'][0].as_py()[:200]
            print(f"    {preview}...")
        print("\n" + "- " * 40)
        
        # JSON
        print("\n[4] JSON (structured document representation)")
        if 'content_json' in result_table.column_names:
            import json as json_lib
            try:
                json_content = json_lib.loads(result_table['content_json'][0].as_py())
                print(f"    Document structure: {list(json_content.keys())[:5]}...")
                print(f"    Preview: {str(json_content)[:200]}...")
            except:
                preview = result_table['content_json'][0].as_py()[:200]
                print(f"    {preview}...")
        print("\n" + "- " * 40)
        
        # DocTags
        print("\n[5] DOCTAGS (Docling native format with semantic tags)")
        if 'content_doctags' in result_table.column_names:
            preview = result_table['content_doctags'][0].as_py()[:200]
            print(f"    {preview}...")
        print("\n" + "- " * 40)
        
        # DocLang
        print("\n[6] DOCLANG (advanced document representation)")
        if 'content_doclang' in result_table.column_names:
            preview = result_table['content_doclang'][0].as_py()[:200]
            print(f"    {preview}...")
    
    print("\n" + "="*80)
    print("\nUse Case Guide:")
    print("  - Markdown: Text processing, chunking, embeddings")
    print("  - HTML: Web display, rich formatting")
    print("  - Plain Text: Simple text analysis, search")
    print("  - JSON: Programmatic access to document structure")
    print("  - DocTags: Semantic document analysis")
    print("  - DocLang: Advanced document processing")
    print("="*80)

# Clean up stored data
cleanup_flow_data(manager)


## 7. Configuration Quick Reference



Quick reference for commonly used ExtractOperator parameters. Copy and modify these snippets as needed.



### Entity Extraction Enhancements

```python
"entity_extraction": {
    "provider": "litellm",
    "expand_extracted_data": true,      # Creates entity_* columns for each field
    "max_doc_chars": 8000,              # Limits document size (cost/performance)
    "output_column": "invoice_data",    # Custom column name
    "provider_config": {
        "temperature": 0.0,             # 0.0=deterministic, 1.0=creative
        "max_tokens": 2000,             # Limit response size
        "timeout": 300                  # Request timeout (seconds)
    }
}
```



### Additional Output Formats

```python
"text_extraction": {
    "provider_config": {
        "additional_formats": ["html", "json", "text", "doctags", "doclang"]
        # Creates: content_html, content_json, content_text, content_doctags, content_doclang
    }
}
```



### Performance Tuning

```python
"config": {
    "max_workers": 4,        # Parallel processing (adjust for CPU/memory)
    "use_processes": false   # true for CPU-intensive tasks
}
```



### Alternative Providers

```python
# Remote processing (scalable)
"text_extraction": {
    "provider": "docling_serve",
    "provider_config": {
        "base_url": "http://localhost:5001",
        "table_mode": "accurate"  # or "fast"
    }
}

# Template-based entity extraction (no LLM)
"entity_extraction": {
    "provider": "docling"
}
```



### VLM Engine Options

```python
"text_extraction": {
    "provider_config": {
        "vlm_pipeline": {
            "engine": "transformers",  # Local models (no API)
            # OR "api_ollama", "api_openai", "mlx"
            "engine_options": {
                "model_id": "microsoft/Florence-2-large"
            }
        }
    }
}
```

**Available VLM Presets:** `granite_docling`, `smoldocling`, `deepseek_ocr`, `qwen`, `pixtral`, `phi4`, `got_ocr`



### Quick Decision Guide

| Need | Use |
|------|-----|
| Individual entity columns | `expand_extracted_data: true` |
| Control LLM costs | `max_doc_chars: 8000` |
| Multiple output formats | `additional_formats: ["html", "json"]` |
| Faster processing | `max_workers: 4+` |
| Scalable extraction | `provider: "docling_serve"` |
| No LLM entity extraction | `provider: "docling"` |

**Full Documentation:** [`docs/operators/extract/extract_operator_config.md`](../../docs/operators/extract/extract_operator_config.md)


## 8. Tips and Best Practices

### 1. Choosing Extraction Provider
```python
# For local processing (no API costs)
"provider": "docling_library"

# For remote processing (scalable)
"provider": "docling_serve"
```

### 2. Entity Extraction Providers
```python
# Local LLM (Ollama)
"provider": "litellm"
"model_id": "ollama/llama3.2"

# Cloud LLM (OpenAI, Anthropic, etc.)
"provider": "litellm"
"model_id": "gpt-4"

# IBM watsonx.ai
"provider": "watsonx"
```

### 3. Performance Optimization
- Use `max_workers` for parallel processing
- Start with small batches for testing
- Cache extraction results when possible
- Use basic extraction for bulk processing

## 9. Troubleshooting

### Common Issues and Solutions

#### 1. Ollama Connection Errors
```
Error: Connection refused to localhost:11434
```
**Solution:**
- Start Ollama: `ollama serve`
- Verify it's running: `curl http://localhost:11434`

#### 2. Model Not Found
```
Error: model 'llama3.2' not found
```
**Solution:**
- Pull the model: `ollama pull llama3.2`
- List available models: `ollama list`

#### 3. Vision Model Issues
```
Error: VLM pipeline failed
```
**Solution:**
- Ensure vision model is installed: `ollama pull ibm/granite-docling:258m`
- Check model supports vision: `ollama show ibm/granite-docling:258m`

#### 4. Memory Issues
```
Error: Out of memory
```
**Solution:**
- Reduce `max_workers` in flow config
- Process documents in smaller batches
- Use `max_doc_chars` to limit document size

#### 5. Slow Extraction
**Solutions:**
- Use basic extraction instead of VLM for bulk processing
- Increase `max_workers` for parallel processing
- Use `docling_serve` for distributed processing

### Getting Help
- Check logs in `./data/` directory
- Review [ExtractOperator documentation](../../docs/operators/extract/)
- Open an issue on GitHub

## Summary

You've learned:
1. ✓ Basic text extraction with Docling
2. ✓ VLM-enhanced extraction for complex layouts
3. ✓ Entity extraction without schema (exploratory)
4. ✓ Entity extraction with schema (production)
5. ✓ Extraction mode comparison and selection
6. ✓ Multi-format output options
7. ✓ Tips and best practices for optimization
8. ✓ Troubleshooting common issues

## Next Steps

- **[04_embeddings_vectordb.ipynb](04_embeddings_vectordb.ipynb)** - Generate embeddings and store in vector DB
- **[05_quality_operators.ipynb](05_quality_operators.ipynb)** - Assess extraction quality
- **[06_rag_pipeline.ipynb](06_rag_pipeline.ipynb)** - Build complete RAG pipeline

## Learn More

- [ExtractOperator Documentation](../../docs/operators/extract/extract_operator_config.md)
- [Docling Documentation](https://github.com/DS4SD/docling)
- [LiteLLM Documentation](https://docs.litellm.ai/)